In [0]:
%sql
-- Weekly sales per product, all stores. A product keeps its original GTIN as product_key across a changeover,
-- so old and new GTIN sales are added together (also used by the Genie space).
CREATE OR REPLACE VIEW ${catalog}.gold.product_weekly_sales
COMMENT 'Weekly unit sales and revenue per product, all stores combined. A product keeps one product_key (its original GTIN) across a GTIN changeover, so old and new GTIN sales are added together. Sales are synthetic demo data.'
AS
WITH changeovers AS (
  SELECT identifier_old AS gtin_old, identifier_new AS gtin_new
  FROM ${catalog}.silver.product_launches
  WHERE identifier_type = 'GTIN' AND identifier_old IS NOT NULL AND identifier_new IS NOT NULL AND identifier_old <> identifier_new
)
SELECT
  COALESCE(c.gtin_old, s.gtin) AS product_key,
  MIN(s.client_id) AS client_id,
  MIN(s.article_description) AS article_description,
  MIN(s.category) AS category,
  s.week,
  SUM(s.units) AS units,
  ROUND(SUM(s.revenue_eur), 2) AS revenue_eur,
  ROUND(AVG(CAST(s.on_promo AS INT)), 2) AS promo_share,
  array_join(array_sort(collect_set(s.gtin)), ', ') AS gtins_sold
FROM ${catalog}.silver.weekly_sales s
LEFT JOIN changeovers c ON c.gtin_new = s.gtin
GROUP BY COALESCE(c.gtin_old, s.gtin), s.week;

-- Cutoff = last week with sales, but never past the last complete Monday-week
CREATE OR REPLACE TEMPORARY VIEW sales_params AS
SELECT LEAST(MAX(week), date_sub(CAST(DATE_TRUNC('week', current_date()) AS DATE), 7)) AS cutoff, 12 AS horizon_weeks
FROM ${catalog}.gold.product_weekly_sales;

CREATE OR REPLACE TEMPORARY VIEW sales_history AS
SELECT product_key, CAST(week AS TIMESTAMP) AS ds, CAST(units AS DOUBLE) AS units
FROM ${catalog}.gold.product_weekly_sales, sales_params WHERE week <= cutoff;

CREATE OR REPLACE TEMPORARY VIEW product_attributes AS
SELECT product_key, MIN(client_id) AS client_id, MIN(article_description) AS article_description, MIN(category) AS category,
       max_by(gtins_sold, week) AS current_gtin
FROM ${catalog}.gold.product_weekly_sales GROUP BY product_key;

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW sales_fc AS
SELECT * FROM ai_forecast(
  TABLE(sales_history),
  horizon => (SELECT CAST(date_add(cutoff, 7 * horizon_weeks + 1) AS TIMESTAMP) FROM sales_params),  -- right-exclusive
  time_col => 'ds', value_col => 'units', group_col => 'product_key',
  prediction_interval_width => 0.90, frequency => 'W-MON', parameters => '{"global_floor": 0}'
);

CREATE TABLE IF NOT EXISTS ${catalog}.gold.sales_forecast_ai (
  model STRING, product_key STRING, current_gtin STRING, client_id STRING, article_description STRING, category STRING,
  week DATE, forecast_units DOUBLE, lower_units DOUBLE, upper_units DOUBLE, generated_at TIMESTAMP
) COMMENT 'ai_forecast forecast of weekly unit sales per product (12 weeks, 90% range). Written by src/forecasting/run_sales_forecast.';

INSERT OVERWRITE ${catalog}.gold.sales_forecast_ai
SELECT 'ai_forecast', f.product_key, a.current_gtin, a.client_id, a.article_description, a.category, CAST(f.ds AS DATE),
       GREATEST(f.units_forecast, 0), GREATEST(f.units_lower, 0), GREATEST(f.units_upper, f.units_forecast, 0), current_timestamp()
FROM sales_fc f JOIN product_attributes a USING (product_key);

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW sales_fc_holdout AS
SELECT * FROM ai_forecast(
  TABLE(SELECT * FROM sales_history WHERE CAST(ds AS DATE) <= (SELECT date_sub(cutoff, 7 * horizon_weeks) FROM sales_params)),
  horizon => (SELECT CAST(date_add(cutoff, 1) AS TIMESTAMP) FROM sales_params),
  time_col => 'ds', value_col => 'units', group_col => 'product_key',
  prediction_interval_width => 0.90, frequency => 'W-MON', parameters => '{"global_floor": 0}'
);

CREATE TABLE IF NOT EXISTS ${catalog}.gold.sales_forecast_accuracy (
  model STRING, holdout_start DATE, holdout_end DATE, products BIGINT, points BIGINT,
  wape_pct DOUBLE, mae_units DOUBLE, coverage_pct DOUBLE, bias_pct DOUBLE, evaluated_at TIMESTAMP
) COMMENT 'Accuracy of each sales forecast model on the last 12 weeks before the cutoff (forecast made without them). WAPE = total absolute error / total actual units.';

DELETE FROM ${catalog}.gold.sales_forecast_accuracy WHERE model = 'ai_forecast';
INSERT INTO ${catalog}.gold.sales_forecast_accuracy
SELECT 'ai_forecast', MIN(CAST(h.ds AS DATE)), MAX(CAST(h.ds AS DATE)), COUNT(DISTINCT h.product_key), COUNT(*),
       SUM(ABS(a.units - h.units_forecast)) / SUM(a.units) * 100,
       AVG(ABS(a.units - h.units_forecast)),
       AVG(CASE WHEN a.units BETWEEN h.units_lower AND h.units_upper THEN 1 ELSE 0 END) * 100,
       (SUM(h.units_forecast) - SUM(a.units)) / SUM(a.units) * 100,
       current_timestamp()
FROM sales_fc_holdout h
JOIN sales_history a ON a.product_key = h.product_key AND a.ds = h.ds;

SELECT * FROM ${catalog}.gold.sales_forecast_accuracy;